# Model Evaluation: LSTM vs CNN-LSTM vs SVM

This notebook evaluates trained EMG-ASL models against held-out synthetic data
and reports accuracy, per-class breakdown, confusion matrix, inference latency,
and model file sizes.

**Before running:** make sure you have trained the baseline model:
```bash
python scripts/train_synthetic_baseline.py
```

The notebook assumes it is run from the `notebooks/` directory. All paths
use `..` to reference the project root.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import confusion_matrix, classification_report
import torch

from src.utils.constants import ASL_LABELS, N_CHANNELS, SAMPLE_RATE
from src.data.synthetic import generate_dataset
from src.data.loader import create_windows
from src.models.lstm_classifier import ASLEMGClassifier

print(f"ASL vocabulary size : {len(ASL_LABELS)} classes")
print(f"EMG channels        : {N_CHANNELS}")
print(f"Sample rate         : {SAMPLE_RATE} Hz")
print(f"Labels              : {ASL_LABELS}")

## 1. Load Models and Data

We load the exported ONNX model from `models/asl_emg_classifier.onnx` and
generate a held-out synthetic evaluation set with a fixed seed so results
are reproducible across runs.

In [ ]:
import onnxruntime as ort

# Load baseline ONNX model
MODEL_PATH = '../models/asl_emg_classifier.onnx'
if not os.path.exists(MODEL_PATH):
    print("Model not found. Run: python scripts/train_synthetic_baseline.py")
    sess = None
else:
    sess = ort.InferenceSession(MODEL_PATH)
    inp = sess.get_inputs()[0]
    print(f"Model loaded successfully")
    print(f"  Input name  : {inp.name}")
    print(f"  Input shape : {inp.shape}")
    print(f"  Input dtype : {inp.type}")

# Generate eval dataset (3 synthetic participants, fixed seed for reproducibility)
print("\nGenerating synthetic evaluation dataset (3 participants, seed=999)...")
eval_dfs = generate_dataset(n_participants=3, seed=999)
print(f"Generated {len(eval_dfs)} participant DataFrames")
for i, df in enumerate(eval_dfs):
    print(f"  Participant {i+1}: {len(df)} rows, columns={list(df.columns[:4])} ...")

In [ ]:
# Collect windows from all 3 participants
all_windows, all_labels = [], []
for df in eval_dfs:
    w, l = create_windows(df)
    all_windows.append(w)
    all_labels.append(l)

windows = np.concatenate(all_windows)  # (N, 40, 8)
labels  = np.concatenate(all_labels)

print(f"Total windows : {len(windows)}")
print(f"Window shape  : {windows.shape}")
print(f"Label count   : {len(labels)}")
print(f"Unique labels : {sorted(set(labels))}")

if sess is None:
    raise RuntimeError("ONNX session not loaded. Train the model first.")

# Flatten for ONNX: (N, 40, 8) -> (N, 320)
flat = windows.reshape(len(windows), -1).astype(np.float32)
print(f"\nFlattened input shape: {flat.shape}")

# Run ONNX inference in batches of 64
preds = []
for i in range(0, len(flat), 64):
    batch = flat[i:i+64]
    out   = sess.run(None, {sess.get_inputs()[0].name: batch})[0]
    preds.extend(np.argmax(out, axis=1))

pred_labels = [ASL_LABELS[p] for p in preds]
true_labels  = list(labels)

print(f"\nInference complete: {len(preds)} predictions")

## 2. Accuracy and Classification Report

Overall accuracy and a full per-class breakdown (precision, recall, F1).

> **Expected result for the synthetic baseline:** ~3% overall accuracy.
> The model weights are randomly initialized and the synthetic data has no
> discriminative signal; this is the chance level for 36 equally distributed
> classes (1/36 = 2.8%). Any score well above 3% after real training indicates
> genuine learning.

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(true_labels, pred_labels)
print(f"Overall Accuracy: {acc:.1%}")
print(f"Chance baseline : {1/len(ASL_LABELS):.1%} (1 / {len(ASL_LABELS)} classes)")
print()
print(classification_report(
    true_labels,
    pred_labels,
    target_names=sorted(set(true_labels)),
    zero_division=0
))

## 3. Confusion Matrix

Row-normalized confusion matrix (each row sums to 1.0).
The diagonal shows per-class recall.

For the synthetic baseline you will see near-uniform values across each row,
reflecting random predictions. After real training, the diagonal should
dominate.

In [ ]:
unique_labels = sorted(set(true_labels))
cm      = confusion_matrix(true_labels, pred_labels, labels=unique_labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Normalized recall')

ax.set_xticks(range(len(unique_labels)))
ax.set_yticks(range(len(unique_labels)))
ax.set_xticklabels(unique_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(unique_labels, fontsize=9)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title(
    f'Confusion Matrix (Accuracy: {acc:.1%})\n'
    'Note: Baseline model trained on synthetic data only',
    fontsize=12
)

plt.tight_layout()
plt.show()

## 4. Per-Class Accuracy

The bar chart below shows per-class recall (diagonal of the normalized
confusion matrix). Colors indicate:

- **Green:** recall >= 50% -- model is getting more than half of this class right
- **Orange:** 25% <= recall < 50% -- marginal; needs more training data
- **Red:** recall < 25% -- model struggles with this class

The dashed navy line shows the mean overall accuracy for reference.

In [ ]:
per_class_acc = cm_norm.diagonal()

colors = [
    'green'  if a >= 0.50 else
    'orange' if a >= 0.25 else
    'red'
    for a in per_class_acc
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(unique_labels, per_class_acc, color=colors)
ax.axhline(y=acc, color='navy', linestyle='--', linewidth=1.5,
           label=f'Mean accuracy: {acc:.1%}')
ax.set_ylim(0, 1.05)
ax.set_xlabel('ASL Class', fontsize=11)
ax.set_ylabel('Recall (per-class accuracy)', fontsize=11)
ax.set_title(
    'Per-Class Accuracy  (green >= 50%, orange >= 25%, red < 25%)',
    fontsize=12
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Summary counts
n_green  = sum(1 for a in per_class_acc if a >= 0.50)
n_orange = sum(1 for a in per_class_acc if 0.25 <= a < 0.50)
n_red    = sum(1 for a in per_class_acc if a < 0.25)
print(f"Green  (>= 50%): {n_green} classes")
print(f"Orange (>= 25%): {n_orange} classes")
print(f"Red    (< 25%) : {n_red} classes")

## 5. Inference Latency

We benchmark ONNX Runtime on a single-window input (320-dim float32).
The target is **< 20 ms per window**, which comfortably fits inside the
100 ms window step (200 ms window at 50% overlap = new window every 100 ms
at 200 Hz).

We run 10 warmup iterations to let ONNX Runtime JIT any kernel selection,
then measure 100 iterations and report mean, P50, P95, and P99.

In [ ]:
import time

single_input = flat[:1]  # shape (1, 320)
input_name   = sess.get_inputs()[0].name

# Warmup -- ONNX Runtime may need a few runs to select optimal kernels
for _ in range(10):
    sess.run(None, {input_name: single_input})

# Benchmark
times = []
for _ in range(100):
    start = time.perf_counter()
    sess.run(None, {input_name: single_input})
    times.append((time.perf_counter() - start) * 1000)  # ms

TARGET_MS = 20.0
mean_ms   = np.mean(times)

print("ONNX inference latency (single window, 320-dim input):")
print(f"  Mean : {mean_ms:.2f} ms")
print(f"  P50  : {np.percentile(times, 50):.2f} ms")
print(f"  P95  : {np.percentile(times, 95):.2f} ms")
print(f"  P99  : {np.percentile(times, 99):.2f} ms")
print()
print(f"Target: < {TARGET_MS:.0f} ms  (one window every 100 ms at 200 Hz, 50% overlap)")
status = 'PASS' if mean_ms < TARGET_MS else 'FAIL'
print(f"Status: {status}")

# Latency distribution plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(times, bins=30, color='steelblue', edgecolor='white')
ax.axvline(TARGET_MS, color='red', linestyle='--', label=f'Target {TARGET_MS:.0f} ms')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('ONNX Single-Window Inference Latency Distribution (n=100)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Model Size

For on-device iOS deployment the ONNX model must be under 25 MB to remain
within App Store binary delta limits. The LSTM baseline is typically well
under 5 MB; this cell confirms the actual sizes on disk.

In [ ]:
model_paths = [
    '../models/asl_emg_classifier.onnx',
    '../models/asl_emg_classifier.pt',
]

print("Model file sizes:")
for model_path in model_paths:
    if os.path.exists(model_path):
        size_bytes = os.path.getsize(model_path)
        size_mb    = size_bytes / 1e6
        flag       = '  OK' if size_mb < 25 else '  WARN: exceeds 25 MB App Store limit'
        print(f"  {os.path.basename(model_path):40s} {size_mb:6.2f} MB{flag}")
    else:
        print(f"  {os.path.basename(model_path):40s} not found")

print()
print("On-device deployment notes (iPhone via Core ML / ONNX Runtime Mobile):")
print("  ONNX model must be < 25 MB for App Store delta updates")
print("  Current LSTM baseline is typically < 5 MB")
print("  Use ONNX quantization (INT8) to halve size if needed:")
print("    python scripts/quantize_onnx.py --input models/asl_emg_classifier.onnx")